In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import lightgbm as lgb

### LightGBM vs XGBoost

**LightGBM** (Light Gradient Boosting Machine) is a gradient boosting framework developed by Microsoft. It's designed to be:

| Feature | LightGBM | XGBoost |
|---------|----------|---------|
| **Speed** | ⚡ Faster (up to 10x) | Slower |
| **Memory** | 💾 Lower memory usage | Higher memory usage |
| **Accuracy** | 🎯 Comparable or better | Comparable |
| **Tree Growth** | 🌿 Leaf-wise (vertical) | 🌳 Level-wise (horizontal) |
| **Best For** | Large datasets, fast prototyping | Small datasets, high precision |

| Aspect | XGBoost (Level-wise) | LightGBM (Leaf-wise) |
|--------|----------------------|---------------------|
| **Growth** | Even, balanced (Grows evenly) | Asymmetrical, unbalanced (Grows asymmetrically) |
| **Speed** | Slower | Faster (up to 10x) |
| **Risk** | Safer | Can overfit if not controlled |
| **Best For** | Small datasets, high precision | Large datasets, fast training |

In [3]:
data = pd.read_csv('../datasets/titanic.csv')
data.columns = [col.lower() for col in data.columns]
data

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [16]:
# data
data['title'] = data['name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())

# Group rare titles
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Major': 'Rare',
    'Mlle': 'Miss',
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Lady': 'Rare',
    'Sir': 'Rare',
    'Countess': 'Rare',
    'Jonkheer': 'Rare',
    'Don': 'Rare'
}
data['title'] = data['title'].map(title_mapping)
data['family_size'] = data['sibsp'] + data['parch'] + 1
data['is_alone'] =  (data['family_size'] == 1).astype(int)
data['cabin_prefix'] = data['cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'Unknown') # cabin prefix
data['age_group'] = pd.cut(data['age'], bins=[0, 12, 18, 30, 50, 100], labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Elder'])
data['fare_group'] = pd.cut(data['fare'], bins=[0, 10, 25, 50, 100, 600],labels=['Low', 'Medium', 'High', 'Very High', 'Luxury'])

features = [
    'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
    'embarked', 'title', 'family_size', 'is_alone', 'cabin_prefix'
]

X = data[features]
y = data['survived']

# missing values
X_prepared = X.copy()
X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
X_prepared['fare'].fillna(X_prepared['fare'].median(), inplace=True)
X_prepared['embarked'].fillna(X_prepared['embarked'].mode()[0], inplace=True)

# encode categorical features
le = LabelEncoder()
categorical_cols = ['sex', 'embarked', 'title', 'cabin_prefix']
for col in categorical_cols:
    X_prepared[col] = le.fit_transform(X_prepared[col].astype(str))

# data split
X_train, X_test, y_train, y_test = train_test_split(X_prepared, y, test_size=0.2, random_state=42, stratify=y)

# LightGBM Dataset (to prevent OSError on windows we need to use lgb dataset and numpy arrays)
# better to test on other os and not windows!
X_train_np = X_train.values
X_test_np = X_test.values
y_train_np = y_train.values
y_test_np = y_test.values

dtrain = lgb.Dataset(X_train_np, label=y_train_np)
dtest = lgb.Dataset(X_test_np, label=y_test_np, reference=dtrain)

# parameters
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'max_depth': 5,
    'learning_rate': 0.1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'num_threads': 1  # Force single thread
}

# model
model = lgb.train(
    params=params,
    train_set=dtrain,
    valid_sets=[dtrain, dtest],
    num_boost_round=100,
    callbacks=[lgb.early_stopping(10), lgb.log_evaluation(0)]
)
# model.fit(X_train.values, y_train.values)

# predict
y_pred_proba_train = model.predict(X_train_np, num_iteration=model_native.best_iteration)
y_pred_proba_test = model.predict(X_test_np, num_iteration=model_native.best_iteration)

y_pred = (y_pred_proba_test > 0.5).astype(int)
y_pred_train = (y_pred_proba_train > 0.5).astype(int)

C:\Users\rah\AppData\Local\Temp\ipykernel_9836\2239536743.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
C:\Users\rah\AppData\Local\Temp\ipykernel_9836\2239536743.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

OSError: exception: access violation reading 0x0000000000000000